# DeepNoise: End-to-End Pipeline

Welcome to the master pipeline notebook for the **DeepNoise Project**. This notebook coordinates the entire pipeline from data ingestion to feature engineering, model training, and model evaluation.

### Pipeline Structure
The workflow consists of 5 main stages:
1. **`download_data.py`**: Downloads raw acoustic WAV files from Zenodo.
2. **`preprocess.py`**: Normalizes sample rates, volume amplitude, and pads/truncates files to a standard 4.0 seconds.
3. **`extract_features.py`**: Extracts 2D Mel-spectrograms from the preprocessed audio and saves them as binary NumPy files.
4. **`train.py`**: Trains the custom 2D CNN in native PyTorch with early stopping and class-weighting, saving the best weights.
5. **`evaluate.py`**: Evaluates the trained model on unseen test data, printing classification reports and saving a confusion matrix heatmap.

---

## Step 1: Ingest Raw Dataset

We start by running the data downloader script to fetch the DCASE 2020 machine sound subsets (fan and pump) from Zenodo. The script downloads the zip archives directly, extracts them to `data/raw/`, and deletes the archives afterwards.

In [ ]:
%run download_data.py

## Step 2: Audio Preprocessing

Next, we run the preprocessor. This script standardizes the raw WAV files by:
- Resampling all audio to 22,050 Hz.
- Standardizing duration to 4.0 seconds (padding shorter clips with silence, truncating longer ones).
- Normalizing amplitude levels to a range of [-1.0, 1.0].

Processed waveforms are saved in parallel to `data/processed/` using your CPU cores.

In [ ]:
%run preprocess.py

## Step 3: Feature Extraction (Mel-Spectrograms)

With the standardized WAV files ready, we compute their 2D Mel-spectrogram frequency representations (128 bands by 173 time bins). These are scaled logarithmically (in decibels) and saved as binary NumPy `.npy` files under `data/features/` in parallel.

In [ ]:
%run extract_features.py

## Step 4: CNN Training (Native PyTorch)

Now we train our custom 2D CNN model using the extracted Mel-spectrogram features. The script:
- Loads features directly into system memory.
- Splits the dataset into 70% Train, 15% Validation, and 15% Test sets.
- Runs training on the target device (GPU if available).
- Saves the best model checkpoint to `models/best_cnn.pth` and the unseen test set to `models/test_split.npz`.

In [ ]:
%run train.py

## Step 5: Model Evaluation

Finally, we run the evaluation script to test the model's accuracy, precision, recall, and F1-score on the unseen test split. The script prints a classification report and saves a Confusion Matrix plot to `results/cnn_confusion_matrix.png`.

In [ ]:
%run evaluate.py